# hybridad launcher

Thin launcher around the CLI: every step writes its results to disk, so the notebook holds no hidden state. Run cells top to bottom. On Colab, first mount the drive that holds the datasets and set the paths in cell 2.

In [ ]:
import os, subprocess, sys
PROJECT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT)
IN_COLAB = 'google.colab' in sys.modules
print(PROJECT, 'colab' if IN_COLAB else 'local')

In [ ]:
# Paths: edit for your machine (Colab example shown). These become environment overrides, not file edits.
if IN_COLAB:
    os.environ['HYBRIDAD_MVTEC_AD'] = '/content/drive/MyDrive/data/MVTecAD'
    os.environ['HYBRIDAD_MVTEC_LOCO'] = '/content/drive/MyDrive/data/MVTec_LOCO'
    os.environ['HYBRIDAD_WORK_DIR'] = '/content/hybridad_work'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[faiss,test]'], check=True)

In [ ]:
def cli(*args):
    env = dict(os.environ, PYTHONPATH=os.path.join(PROJECT, 'src'))
    r = subprocess.run([sys.executable, '-m', 'hybridad.cli', *args], env=env, text=True, capture_output=True)
    print(r.stdout[-5000:]); print(r.stderr[-3000:])
    r.check_returncode()

cli('env')

In [ ]:
# Code-correctness checks (synthetic fixture only; NOT benchmark evidence)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], env=dict(os.environ, PYTHONPATH='src'))
cli('smoke', '--experiment', 'experiments/G_full.yaml')

In [ ]:
cli('manifests', '--dataset', 'all')

In [ ]:
# Fit + score + calibrate one category per dataset (no labels read)
cli('prepare', '--experiment', 'experiments/G_full.yaml', '--dataset', 'mvtec_ad', '--category', 'bottle')
cli('prepare', '--experiment', 'experiments/G_full.yaml', '--dataset', 'mvtec_loco', '--category', 'breakfast_box')

In [ ]:
# Pre-register BEFORE test evaluation (write-once). Edit the note.
# cli('preregister', '--experiments', 'experiments/*.yaml', '--dataset', 'all', '--note', 'describe the frozen plan')

In [ ]:
# Final evaluation and report (refused for non-pre-registered configs unless '--exploratory')
# cli('evaluate', '--experiment', 'experiments/G_full.yaml', '--dataset', 'mvtec_ad', '--category', 'bottle')
# cli('report')

In [ ]:
from IPython.display import Markdown, display
p = os.path.join(PROJECT, 'docs', 'RESULTS.md')
display(Markdown(open(p, encoding='utf-8').read()) if os.path.exists(p) else 'no report yet')